In [2]:
import asyncio
import xmlschema
import networkx as nx
import polars as pl
from itertools import zip_longest

In [3]:
from ggblab import GeoGebra
ggb = await GeoGebra().init(use_vscode=True)

Using local cached file: xsd/common.xsd


<IPython.core.display.JSON object>

In [4]:
from ggblab_extra import ConstructionIO
from ggblab_extra import ConstructionTreeParser

In [6]:
%cd examples/

/Users/manabu/work/ggblab/examples


In [5]:
await ggb.function('showMenuBar', [False])

In [6]:
await ggb.function('showToolBar', [False])

In [22]:
await ggb.function('getAllObjectNames')

[]

In [12]:
r = await ggb.function('getXML', ['n'])

In [13]:
print(r)

<element type="numeric" label="n">
	<value val="12"/>
	<slider min="1" max="30" absoluteScreenLocation="true" width="200" x="123" y="145" fixed="false" horizontal="true" showAlgebra="true"/>
	<lineStyle thickness="10" type="0" typeHidden="1"/>
	<show object="true" label="true"/>
	<objColor r="0" g="0" b="0" alpha="0.10000000149011612"/>
	<layer val="0"/>
	<labelMode val="1"/>
	<animation step="1" type="0" playing="false"/>
</element>



In [23]:
xml_str = r'''<element type="numeric" label="n">
	<value val="12"/>
	<slider min="1" max="30" absoluteScreenLocation="true" width="200" x="123" y="145" fixed="false" horizontal="true" showAlgebra="true"/>
	<lineStyle thickness="10" type="0" typeHidden="1"/>
	<show object="true" label="true"/>
	<objColor r="0" g="0" b="0" alpha="0.10000000149011612"/>
	<layer val="0"/>
	<labelMode val="1"/>
	<animation step="1" type="0" playing="false"/>
</element>
'''
xml_str

'<element type="numeric" label="n">\n\t<value val="12"/>\n\t<slider min="1" max="30" absoluteScreenLocation="true" width="200" x="123" y="145" fixed="false" horizontal="true" showAlgebra="true"/>\n\t<lineStyle thickness="10" type="0" typeHidden="1"/>\n\t<show object="true" label="true"/>\n\t<objColor r="0" g="0" b="0" alpha="0.10000000149011612"/>\n\t<layer val="0"/>\n\t<labelMode val="1"/>\n\t<animation step="1" type="0" playing="false"/>\n</element>\n'

In [57]:
r = await ggb.function('getXML', ['n'])
o = ggb.file.ggb_schema.decode(r)
o

{'@type': 'numeric',
 '@label': 'n',
 'value': [{'@val': 4.0}],
 'slider': [{'@min': '0',
   '@max': '7',
   '@absoluteScreenLocation': True,
   '@width': 200.0,
   '@x': 73.0,
   '@y': 99.0,
   '@fixed': False,
   '@horizontal': True,
   '@showAlgebra': True}],
 'lineStyle': [{'@thickness': 10, '@type': 0, '@typeHidden': 1}],
 'show': [{'@object': True, '@label': True}],
 'objColor': [{'@r': 0, '@g': 0, '@b': 0, '@alpha': 0.10000000149011612}],
 'layer': [{'@val': 9}],
 'labelMode': [{'@val': 1}],
 'animation': [{'@step': '1', '@type': 0, '@playing': False}]}

In [58]:
o['slider'][0]['@max'] = '0'
o['slider'][0]['@max'] = '7'
o

{'@type': 'numeric',
 '@label': 'n',
 'value': [{'@val': 4.0}],
 'slider': [{'@min': '0',
   '@max': '7',
   '@absoluteScreenLocation': True,
   '@width': 200.0,
   '@x': 73.0,
   '@y': 99.0,
   '@fixed': False,
   '@horizontal': True,
   '@showAlgebra': True}],
 'lineStyle': [{'@thickness': 10, '@type': 0, '@typeHidden': 1}],
 'show': [{'@object': True, '@label': True}],
 'objColor': [{'@r': 0, '@g': 0, '@b': 0, '@alpha': 0.10000000149011612}],
 'layer': [{'@val': 9}],
 'labelMode': [{'@val': 1}],
 'animation': [{'@step': '1', '@type': 0, '@playing': False}]}

In [59]:
x = xmlschema.etree_tostring(ggb.file.ggb_schema.encode(o, 'element'))
x

'<element type="numeric" label="n">\n    <value val="4.0" />\n    <slider min="0" max="7" absoluteScreenLocation="true" width="200.0" x="73.0" y="99.0" fixed="false" horizontal="true" showAlgebra="true" />\n    <lineStyle thickness="10" type="0" typeHidden="1" />\n    <show object="true" label="true" />\n    <objColor r="0" g="0" b="0" alpha="0.10000000149011612" />\n    <layer val="9" />\n    <labelMode val="1" />\n    <animation step="1" type="0" playing="false" />\n</element>'

In [60]:
r = await ggb.function('evalXML' , [x])
r

In [35]:

df = await ConstructionIO.initialize_dataframe(ggb, use_applet=True)
df

Sequence,Name,Type,Command,Value,Caption,Layer,ShowObject,ShowLabel,Auxiliary
u32,str,str,str,str,str,u32,bool,bool,bool
0,"""n""","""numeric""",null,"""n = 7""",null,9,true,true,false
1,"""A""","""point""",null,"""A = (-8.7577098825514, 8.09069…",null,9,true,true,false
2,"""B""","""point""",null,"""B = (-7.7537611362231, 5.99152…",null,9,true,true,false
3,"""C""","""point""",null,"""C = (-5.4889374044189, 5.94651…",null,9,true,true,false
4,"""f""","""line""","""Line(A, B)""","""f: 2.0991655605045x + 1.003948…",null,0,true,true,false
5,"""g""","""line""","""Line(A, C)""","""g: 2.144180020238x + 3.2687724…",null,0,true,true,false
6,"""m""","""numeric""",null,"""m = 0""",null,9,true,true,false
7,"""h""","""line""","""AngleBisector(f, g)""","""h: 0.753008148849x + 0.6580111…",null,1,true,true,false
8,"""i""","""line""","""AngleBisector(f, g)""","""i: -0.6580111912171x + 0.75300…",null,1,true,true,false


In [36]:
p = ConstructionTreeParser(df)
g1 = p.parse()
nx.write_network_text(g1)

╟── A
╎   ├─╼ f ╾ B
╎   │   ├─╼ h ╾ g
╎   │   │   └─╼ E ╾ j
╎   │   │       ├─╼ p ╾ g
╎   │   │       │   └─╼ I ╾ g
╎   │   │       │       └─╼ d ╾ E
╎   │   │       │           └─╼ Q ╾ r
╎   │   │       │               └─╼ t ╾ N
╎   │   │       ├─╼ q ╾ f
╎   │   │       │   └─╼ H ╾ f
╎   │   │       │       └─╼ c_1 ╾ L
╎   │   │       ├─╼ J ╾ D
╎   │   │       │   └─╼ e ╾ D
╎   │   │       │       ├─╼ K ╾ f
╎   │   │       │       ├─╼ L ╾ f
╎   │   │       │       │   ├─╼ a ╾ C
╎   │   │       │       │   └─╼  ...
╎   │   │       │       ├─╼ M ╾ g
╎   │   │       │       └─╼ N ╾ g
╎   │   │       │           ├─╼ r ╾ B
╎   │   │       │           │   ├─╼ O ╾ c
╎   │   │       │           │   ├─╼ P ╾ c
╎   │   │       │           │   └─╼  ...
╎   │   │       │           └─╼  ...
╎   │   │       └─╼  ...
╎   │   ├─╼ i ╾ g
╎   │   │   └─╼ D ╾ j
╎   │   │       ├─╼ k ╾ g
╎   │   │       │   └─╼ F ╾ g
╎   │   │       │       ├─╼ c ╾ D
╎   │   │       │       │   └─╼  ...
╎   │   │       │  

In [37]:
p.df

Sequence,Name,Type,Command,Value,Caption,Layer,ShowObject,ShowLabel,Auxiliary,DependsOn
u32,str,str,str,str,str,u32,bool,bool,bool,list[str]
0,"""n""","""numeric""",null,"""n = 7""",null,9,true,true,false,[]
1,"""A""","""point""",null,"""A = (-8.7577098825514, 8.09069…",null,9,true,true,false,[]
2,"""B""","""point""",null,"""B = (-7.7537611362231, 5.99152…",null,9,true,true,false,[]
3,"""C""","""point""",null,"""C = (-5.4889374044189, 5.94651…",null,9,true,true,false,[]
4,"""f""","""line""","""Line(A, B)""","""f: 2.0991655605045x + 1.003948…",null,0,true,true,false,"[""A"", ""B""]"
5,"""g""","""line""","""Line(A, C)""","""g: 2.144180020238x + 3.2687724…",null,0,true,true,false,"[""A"", ""C""]"
6,"""m""","""numeric""",null,"""m = 0""",null,9,true,true,false,[]
7,"""h""","""line""","""AngleBisector(f, g)""","""h: 0.753008148849x + 0.6580111…",null,1,true,true,false,"[""A"", ""B"", … ""g""]"
8,"""i""","""line""","""AngleBisector(f, g)""","""i: -0.6580111912171x + 0.75300…",null,1,true,true,false,"[""A"", ""B"", … ""g""]"


In [24]:
await ggb.listen('n')

{}

In [25]:
await ggb.listen('m')

{}

In [26]:
import ipywidgets as widgets
label1 = widgets.Label(value=ggb.comm.shared_objects['n'])
label2 = widgets.Label(value=ggb.comm.shared_objects['m'])
display(label1, label2)

Label(value='n = 7')

Label(value='m = 7')

In [27]:
async def on_shared_update1(changes):
    # await asyncio.sleep(0)
    # label1.value = changes['n']
    n = int(changes['n'].split()[2])
    # await asyncio.sleep(0)
    await ggb.function("setLayerVisible", list(zip_longest(range(8), [True]*n, fillvalue=False)))

In [28]:
ggb.comm.remove_shared_listener(on_shared_update1)
ggb.comm.add_shared_listener(on_shared_update1)

True

In [29]:
async def on_shared_update2(changes):
    # await asyncio.sleep(0)
    # label2.value = changes['m']
    m = int(changes['m'].split()[2])
    n = int(ggb.comm.shared_objects['n'].split()[2])
    l = df.filter(pl.col("Layer") == n)["Name"].to_list()
    # m = int(ggb.comm.shared_objects['m'].split()[2])
    # list(zip_longest(l, [True]*m, fillvalue=False))
    await ggb.function("setVisible", list(zip_longest(l, [True]*m, fillvalue=False)))

In [30]:
ggb.comm.remove_shared_listener(on_shared_update2)
ggb.comm.add_shared_listener(on_shared_update2)

True

In [118]:
ggb.comm.clear_shared_listeners()

0

In [187]:
%cd examples/

/Users/manabu/work/ggblab/examples


In [ ]:
# ggb.file.source_file = 'eg10_slider.ggb'

In [33]:
ggb.file.base64_buffer = await ggb.function("getBase64")

In [34]:
ggb.file.save(overwrite=True)

In [9]:
ggb.file.load('eg10_slider.ggb')

In [10]:
r = await ggb.function("setBase64", [ggb.construction.base64_buffer.decode('utf-8')])

* 原則（教育観）:
    - 目的化: 再現は「結果」ではなく「理解（なぜその操作か）」を目的にする。
    - 予測→検証: 次に何が起きるか予測させてから操作させる。
    - 説明要求: 手順ごとに短い理由説明（1文）を書かせる。
    - 変奏課題: パラメータを少し変えた課題で本質が移るか確認する。
    - 生成的課題: 「同じ発想で別の図形を作る」など転移を問う。
* ggblabで実装できる仕組み（短）:
    - 段階公開（layer slider）: 各レイヤーに「解説」「問い」「期待する操作」を紐付け、スライダーで段階的に提示。
    - 予測プロンプト: 各ステップの前に「次に何が起きる？」を表示し、回答を記録。
    - 説明入力欄: 学生が操作毎に短い説明を入力 → 教師や自動ルールでフィードバック。
    - 変化タスク自動化: DataFrame→コマンド生成を利用してパラメータをランダム化した派生課題を作る。
    - 操作ログ＋解析: 操作順・所要時間・試行回数をログ化して学習診断に使う。
    - 差分フィードバック: 学生構成と模範構成を比較して「次に直すべき一手」を提示。

In [11]:
await ggb.command("Midpoint(D,E)")

'J'

In [12]:
await ggb.command("Circle(J,D)")

'e'

In [15]:
await ggb.command("Line(B, N)")

'r'

In [16]:
await ggb.command("Circle(A,F)")

's'

In [17]:
await ggb.command("Circle(N,Q)")

't'

In [31]:
await ggb.command("Circle(L,H)")

'c_1'

In [32]:
await ggb.command("Line(C, L)")

'a'